# ER-CyRIS — Siklus 1 Confirmatory Multi-Dataset Replication

Notebook ini **bukan pengganti historis** eksperimen Siklus 1 awal yang berpusat pada CICIDS2018.

Tujuannya adalah menambahkan **confirmatory multi-dataset replication** setelah masukan penguji, khususnya untuk:
1. menguji apakah pola weakness Siklus 1 berulang pada beberapa domain/dataset,
2. memperkuat prinsip *experience → evidence → knowledge*,
3. menghindari klaim knowledge-based rule yang hanya berasal dari satu kasus,
4. mempertahankan RM1 dan desain S0–S5 tanpa mengubah kontribusi asli penelitian.

Dataset mengikuti Siklus 2:
- CICIDS2018
- HDFS
- BGL
- UNSW-NB15

Catatan penting:
- Seluruh preprocessing di-fit **hanya pada training partition**.
- Duplikasi identik dibuang sebelum split bila memungkinkan.
- Hasil per dataset dibaca sebagai **evidence lintas-kondisi**, bukan sebagai perbandingan angka mentah antar-domain.
- Notebook ini tidak otomatis menyatakan W1–W6 sebagai "knowledge-based rules". Notebook menghasilkan evidence yang nantinya disintesis oleh peneliti.

In [ ]:
# ============================================================
# 0. INSTALL (jalankan jika diperlukan di Google Colab)
# ============================================================
# !pip -q install xgboost shap tqdm

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
import os, re, json, time, warnings, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.cluster import DBSCAN

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
print("✅ Imports ready")

In [ ]:
# ============================================================
# 2. GOOGLE DRIVE + CONFIG
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Mengikuti folder dataset Siklus 2 agar tidak perlu duplikasi file
BASE = '/content/drive/MyDrive/ercyris_siklus2/'

CFG = {
    'CICIDS': BASE + '02-14-2018.csv',
    'HDFS':   BASE + 'HDFS.log',
    'HLABEL': BASE + 'anomaly_label.csv',
    'BGL':    BASE + 'BGL.log',
    'UNSW':   BASE + 'UNSW-NB15_1.csv',

    'SEED': 42,
    'TEST_SIZE': 0.20,
    'MAX_CICIDS': 500_000,
    'MAX_BGL': 50_000,

    # S1 imbalance — minority:majority target ratios
    'S1_RATIOS': [1.0, 0.5, 0.2, 0.1],

    # S2 perturbation
    'S2_NOISE_SIGMAS': [0.01, 0.05, 0.10],
    'S2_MISSING_RATES': [0.01, 0.03, 0.05],

    # S4 sensitivity
    'IF_CONTAMS': [0.01, 0.03, 0.05, 0.10],
    'LOF_K': [10, 20, 50, 100],
    'DBSCAN_Q': [0.90, 0.95, 0.99],

    # S5 latency
    'BATCH_SIZES': [1, 16, 64, 256, 512],

    # SHAP confirmatory stability audit
    'SHAP_N': 500,
    'SHAP_K': 10,
    'SHAP_SEEDS': [42, 52, 62, 72, 82],
    'SHAP_NOISE_SIGMA': 0.01,

    'OUTPUT_DIR': '/content/drive/MyDrive/ercyris_siklus1_confirmatory/'
}

Path(CFG['OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)
print("✅ Config ready")

## 3. Loader Dataset

Loader mengembalikan:
- `X`: DataFrame fitur
- `y`: label biner (0 normal, 1 anomaly/attack)
- `time_key`: urutan temporal/urutan asli untuk S3 bila tersedia
- `meta`: informasi dataset

**Catatan metodologis:** HDFS, BGL, CICIDS2018, dan UNSW-NB15 memiliki unit analisis berbeda. Karena itu hasil tidak dibandingkan sebagai angka absolut antar-dataset. Yang dibandingkan adalah **pola weakness dan arah degradasi**.

In [ ]:
# ============================================================
# 3A. HELPER
# ============================================================
def clean_cols(cols):
    out = []
    for c in cols:
        c = str(c).strip().lower()
        c = c.replace(' ', '_').replace('/', '_per_')
        c = re.sub(r'[^a-z0-9_]+', '_', c)
        out.append(c.strip('_'))
    return out

def dedup_by_features(X, y, time_key=None):
    X = X.copy()
    X.columns = clean_cols(X.columns)
    tmp = X.copy()
    tmp['__y__'] = np.asarray(y)
    if time_key is not None:
        tmp['__time__'] = np.asarray(time_key)

    feat_cols = list(X.columns)
    before = len(tmp)

    # Bila fitur identik tetapi label berbeda, jangan diam-diam pilih satu:
    # kelompok konflik dihapus dari confirmatory analysis.
    grp = tmp.groupby(feat_cols, dropna=False)['__y__'].nunique()
    conflict_idx = grp[grp > 1].index
    if len(conflict_idx) > 0:
        conflict_keys = set(conflict_idx.tolist())
        key_rows = list(map(tuple, tmp[feat_cols].itertuples(index=False, name=None)))
        keep = np.array([k not in conflict_keys for k in key_rows])
        tmp = tmp.loc[keep].copy()

    tmp = tmp.drop_duplicates(subset=feat_cols, keep='first').reset_index(drop=True)

    y2 = tmp.pop('__y__').astype(int).values
    t2 = tmp.pop('__time__').values if '__time__' in tmp.columns else np.arange(len(tmp))
    X2 = tmp

    print(f"   dedup/conflict audit: {before:,} → {len(X2):,}")
    return X2, y2, t2

def binary_label_from_text(s):
    s = str(s).strip().upper()
    return 0 if 'BENIGN' in s or s in {'NORMAL','0','FALSE'} else 1

In [ ]:
# ============================================================
# 3B. CICIDS2018
# ============================================================
def load_cicids():
    print("📂 CICIDS2018")
    df = pd.read_csv(CFG['CICIDS'], low_memory=False)
    df.columns = clean_cols(df.columns)

    label_col = next(c for c in df.columns if 'label' in c)
    y = df[label_col].map(binary_label_from_text).values

    # simpan timestamp/order sebelum drop
    if 'timestamp' in df.columns:
        t = pd.to_datetime(df['timestamp'], errors='coerce')
        # fallback ke row order untuk timestamp invalid
        time_key = t.astype('int64', errors='ignore') if t.notna().any() else np.arange(len(df))
    else:
        time_key = np.arange(len(df))

    drop = [label_col]
    drop += [c for c in df.columns if c in {
        'flow_id','source_ip','destination_ip','src_ip','dst_ip','simillarhttp','timestamp'
    }]
    X = df.drop(columns=list(set(drop)), errors='ignore')
    X = X.replace([np.inf, -np.inf], np.nan)

    # deterministic cap tetapi urutan asli tetap dipertahankan
    if len(X) > CFG['MAX_CICIDS']:
        rng = np.random.RandomState(CFG['SEED'])
        idx = np.sort(rng.choice(len(X), CFG['MAX_CICIDS'], replace=False))
        X = X.iloc[idx].reset_index(drop=True)
        y = y[idx]
        time_key = np.asarray(time_key)[idx]

    X, y, time_key = dedup_by_features(X, y, time_key)

    print(f"   {len(X):,} rows | attack={y.sum():,} ({100*y.mean():.2f}%)")
    return X, y, time_key, {'dataset':'CICIDS2018','unit':'network flow'}

In [ ]:
# ============================================================
# 3C. HDFS
# ============================================================
def load_hdfs():
    print("📂 HDFS")
    LP = re.compile(r'^(\d{6})\s+(\d{6})\s+\d+\s+(\w+)\s+([^:]+):\s+(.*)')
    BP = re.compile(r'(blk_-?\d+)')
    SM = {'DEBUG':0,'INFO':1,'WARN':2,'WARNING':2,'ERROR':3,'FATAL':4}

    bd = defaultdict(lambda:{
        'n':0,'e':0,'w':0,'comp':set(),'ms':0,'ss':0,'first_seq':10**18
    })

    seq = 0
    with open(CFG['HDFS'], 'r', errors='replace') as f:
        for line in tqdm(f, desc='parse HDFS', leave=False):
            seq += 1
            m = LP.match(line)
            if not m:
                continue
            _,_,lv,comp,msg = m.groups()
            sev = SM.get(lv.upper(),1)
            for blk in BP.findall(msg):
                d = bd[blk]
                d['n'] += 1
                d['comp'].add(comp.strip())
                d['ms'] = max(d['ms'], sev)
                d['ss'] += sev
                d['first_seq'] = min(d['first_seq'], seq)
                if sev >= 3: d['e'] += 1
                if sev == 2: d['w'] += 1

    labels = pd.read_csv(CFG['HLABEL'])
    lmap = dict(zip(
        labels['BlockId'].astype(str),
        (labels['Label'].str.upper() == 'ANOMALY').astype(int)
    ))

    rows, ys, times = [], [], []
    for blk, d in bd.items():
        if blk not in lmap:
            continue
        rows.append({
            'n_events': d['n'],
            'n_error': d['e'],
            'n_warn': d['w'],
            'n_comp': len(d['comp']),
            'max_sev': d['ms'],
            'mean_sev': d['ss']/max(d['n'],1),
            'error_ratio': d['e']/max(d['n'],1),
            'warn_ratio': d['w']/max(d['n'],1)
        })
        ys.append(lmap[blk])
        times.append(d['first_seq'])

    X = pd.DataFrame(rows)
    y = np.asarray(ys, dtype=int)
    time_key = np.asarray(times)

    X, y, time_key = dedup_by_features(X, y, time_key)
    print(f"   {len(X):,} blocks | anomaly={y.sum():,} ({100*y.mean():.2f}%)")
    return X, y, time_key, {'dataset':'HDFS','unit':'block/session aggregation'}

In [ ]:
# ============================================================
# 3D. BGL
# ============================================================
def load_bgl():
    print("📂 BGL")
    SM = {
        'INFO':1,'DEBUG':0,'WARN':2,'WARNING':2,'ERROR':3,'SEVERE':4,
        'FATAL':4,'APPFPO':3,'KERN':2,'KERNSVC':3,'APPFPW':2
    }
    AKW = ['fatal','error','fail','exception','abort','illegal',
           'segfault','panic','crash','out of memory','overflow']

    rows, ys, times = [], [], []
    with open(CFG['BGL'], 'r', errors='replace') as f:
        for seq, line in enumerate(tqdm(f, desc='parse BGL', leave=False)):
            p = line.split(None, 8)
            if len(p) < 9:
                continue

            y = 0 if p[0] == '-' else 1
            sev = SM.get(p[6].upper(),1)
            msg = p[8].lower()

            try:
                hour = int(p[3].split(':')[0]) if ':' in p[3] else 12
            except:
                hour = 12

            rows.append({
                'sev': sev,
                'is_err': int(sev >= 3),
                'is_warn': int(sev == 2),
                'hour': hour,
                'off_hr': int(hour < 6),
                'peak_hr': int(8 <= hour <= 18),
                'err_kw': int(any(k in msg for k in AKW)),
                'has_num': int(bool(re.search(r'\b\d{4,}\b', msg))),
                'msg_len': min(len(msg), 500),
                'comp_len': len(p[5]),
                'node_h': abs(hash(p[4])) % 1000
            })
            ys.append(y)
            times.append(seq)

    X = pd.DataFrame(rows)
    y = np.asarray(ys, dtype=int)
    time_key = np.asarray(times)

    if len(X) > CFG['MAX_BGL']:
        # stratified sample, kemudian urutkan kembali berdasarkan urutan asli
        sel = []
        rng = np.random.RandomState(CFG['SEED'])
        for cls in [0,1]:
            idx = np.where(y == cls)[0]
            n = max(1, int(CFG['MAX_BGL'] * len(idx) / len(y)))
            sel.extend(rng.choice(idx, min(n, len(idx)), replace=False).tolist())
        sel = np.array(sorted(sel))
        X = X.iloc[sel].reset_index(drop=True)
        y = y[sel]
        time_key = time_key[sel]

    X, y, time_key = dedup_by_features(X, y, time_key)
    print(f"   {len(X):,} rows | anomaly={y.sum():,} ({100*y.mean():.2f}%)")
    return X, y, time_key, {'dataset':'BGL','unit':'log event'}

In [ ]:
# ============================================================
# 3E. UNSW-NB15
# ============================================================
def load_unsw():
    print("📂 UNSW-NB15")
    df = pd.read_csv(CFG['UNSW'], low_memory=False)
    df.columns = clean_cols(df.columns)

    label_col = next((c for c in ['label','is_attack'] if c in df.columns), None)
    if label_col is None:
        raise ValueError("UNSW label column tidak ditemukan.")

    y = (pd.to_numeric(df[label_col], errors='coerce').fillna(0) != 0).astype(int).values

    # stime/ltime dipertahankan sebagai time_key tetapi tidak dijadikan predictor mentah
    if 'stime' in df.columns:
        time_key = pd.to_numeric(df['stime'], errors='coerce').fillna(pd.Series(np.arange(len(df)))).values
    else:
        time_key = np.arange(len(df))

    X = df.drop(columns=[
        label_col, 'attack_cat', 'srcip', 'dstip', 'src_ip', 'dst_ip',
        'stime', 'ltime'
    ], errors='ignore')
    X = X.replace([np.inf, -np.inf], np.nan)

    X, y, time_key = dedup_by_features(X, y, time_key)
    print(f"   {len(X):,} rows | attack={y.sum():,} ({100*y.mean():.2f}%)")
    return X, y, time_key, {'dataset':'UNSW-NB15','unit':'network flow'}

In [ ]:
# ============================================================
# 4. LOAD ALL AVAILABLE DATASETS
# ============================================================
LOADERS = {
    'CICIDS2018': (load_cicids, [CFG['CICIDS']]),
    'HDFS':       (load_hdfs, [CFG['HDFS'], CFG['HLABEL']]),
    'BGL':        (load_bgl, [CFG['BGL']]),
    'UNSW-NB15':  (load_unsw, [CFG['UNSW']]),
}

DATASETS = {}

for name, (loader, reqs) in LOADERS.items():
    if all(os.path.exists(p) for p in reqs):
        try:
            DATASETS[name] = loader()
        except Exception as e:
            print(f"❌ {name} gagal: {e}")
    else:
        print(f"⚠️ {name} dilewati; file belum tersedia.")

print("\n✅ Loaded:", list(DATASETS.keys()))

## 5. Train-only preprocessing dan anti-leakage

Untuk menghindari leakage:
- split dilakukan sebelum imputasi/scaling/encoding,
- seluruh transformer di-fit hanya pada training,
- fitur kategorikal pada UNSW menggunakan `OneHotEncoder(handle_unknown='ignore')`,
- timestamp/raw identifiers tidak dipakai sebagai predictor.

In [ ]:
# ============================================================
# 5. PREPROCESSOR + METRICS
# ============================================================
def make_preprocessor(Xtr):
    num_cols = list(Xtr.select_dtypes(include=np.number).columns)
    cat_cols = [c for c in Xtr.columns if c not in num_cols]

    num_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', MinMaxScaler())
    ])

    cat_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    pre = ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols),
    ], remainder='drop')

    return pre, num_cols, cat_cols

def calc_metrics(y_true, y_pred, score=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    far = fp / max(fp + tn, 1)
    out = {
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'far': far,
        'alert_rate': float(np.mean(y_pred)),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)
    }
    if score is not None:
        try:
            out['pr_auc'] = average_precision_score(y_true, score)
        except:
            out['pr_auc'] = np.nan
    else:
        out['pr_auc'] = np.nan
    return out

def fit_supervised(Xtr, ytr):
    return {
        'XGBoost': XGBClassifier(
            random_state=CFG['SEED'], n_jobs=-1,
            eval_metric='logloss', verbosity=0
        ).fit(Xtr, ytr),
        'RandomForest': RandomForestClassifier(
            n_estimators=300, random_state=CFG['SEED'], n_jobs=-1
        ).fit(Xtr, ytr)
    }

def eval_supervised(model, Xte, yte):
    yp = model.predict(Xte)
    score = model.predict_proba(Xte)[:,1] if hasattr(model, 'predict_proba') else None
    return calc_metrics(yte, yp, score)

In [ ]:
# ============================================================
# 6. UNSUPERVISED BASELINES — HELD-OUT SAFE
# ============================================================
def eval_iforest(Xtr, Xte, yte, contamination=0.05):
    m = IsolationForest(
        contamination=contamination,
        random_state=CFG['SEED'], n_jobs=-1
    ).fit(Xtr)
    pred = (m.predict(Xte) == -1).astype(int)
    score = -m.decision_function(Xte)
    return calc_metrics(yte, pred, score)

def eval_lof(Xtr, Xte, yte, k=20, contamination=0.05):
    # novelty=True wajib untuk held-out evaluation
    k = min(k, max(2, len(Xtr)-1))
    m = LocalOutlierFactor(
        n_neighbors=k, novelty=True,
        contamination=contamination, n_jobs=-1
    ).fit(Xtr)
    pred = (m.predict(Xte) == -1).astype(int)
    score = -m.decision_function(Xte)
    return calc_metrics(yte, pred, score)

def eval_dbscan_inductive(Xtr, Xte, yte, quantile=0.95, min_samples=10, max_fit=30000):
    '''
    DBSCAN sklearn tidak memiliki predict() untuk data baru.
    Untuk held-out evaluation digunakan aturan nearest-core-distance:
    point dianggap outlier bila jarak ke core training terdekat > eps.

    Ini adalah confirmatory inductive proxy dan HARUS dilaporkan sebagai demikian.
    '''
    rng = np.random.RandomState(CFG['SEED'])
    if len(Xtr) > max_fit:
        idx = rng.choice(len(Xtr), max_fit, replace=False)
        Xfit = Xtr[idx]
    else:
        Xfit = Xtr

    k = min(10, max(2, len(Xfit)-1))
    nn = NearestNeighbors(n_neighbors=k, n_jobs=-1).fit(Xfit)
    d, _ = nn.kneighbors(Xfit)
    kth = d[:, -1]
    eps = float(np.quantile(kth, quantile))

    db = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit(Xfit)

    if len(db.core_sample_indices_) == 0:
        pred = np.ones(len(Xte), dtype=int)
        score = np.ones(len(Xte))
        return calc_metrics(yte, pred, score), eps, 0

    core = Xfit[db.core_sample_indices_]
    core_nn = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(core)
    dte, _ = core_nn.kneighbors(Xte)
    score = dte[:,0]
    pred = (score > eps).astype(int)

    return calc_metrics(yte, pred, score), eps, len(core)

## 7. Scenario S0 — Baseline

S0 dilakukan pada setiap dataset dengan train/test stratified split.  
Untuk S3 nanti dibuat split temporal terpisah; S0 tetap dipertahankan sebagai baseline yang mudah direplikasi.

In [ ]:
# ============================================================
# 7. S0 BASELINE
# ============================================================
RESULTS = []
CACHE = {}

for ds_name, (X, y, time_key, meta) in DATASETS.items():
    print(f"\n{'='*70}\nS0 — {ds_name}\n{'='*70}")

    idx = np.arange(len(X))
    tr_idx, te_idx = train_test_split(
        idx, test_size=CFG['TEST_SIZE'],
        random_state=CFG['SEED'], stratify=y
    )

    Xtr_raw, Xte_raw = X.iloc[tr_idx].copy(), X.iloc[te_idx].copy()
    ytr, yte = y[tr_idx], y[te_idx]

    pre, _, _ = make_preprocessor(Xtr_raw)
    Xtr = pre.fit_transform(Xtr_raw)
    Xte = pre.transform(Xte_raw)

    models = fit_supervised(Xtr, ytr)

    for mn, m in models.items():
        r = eval_supervised(m, Xte, yte)
        RESULTS.append({'dataset':ds_name,'scenario':'S0','model':mn, **r})
        print(mn, r)

    contam = float(np.clip(ytr.mean(), 0.01, 0.20))

    r = eval_iforest(Xtr, Xte, yte, contam)
    RESULTS.append({'dataset':ds_name,'scenario':'S0','model':'IsolationForest', **r})

    r = eval_lof(Xtr, Xte, yte, 20, contam)
    RESULTS.append({'dataset':ds_name,'scenario':'S0','model':'LOF', **r})

    r, eps, ncore = eval_dbscan_inductive(Xtr, Xte, yte, 0.95, 10)
    RESULTS.append({
        'dataset':ds_name,'scenario':'S0','model':'DBSCAN_inductive',
        'eps':eps,'n_core':ncore, **r
    })

    CACHE[ds_name] = {
        'X':X, 'y':y, 'time_key':np.asarray(time_key),
        'tr_idx':tr_idx,'te_idx':te_idx,
        'Xtr_raw':Xtr_raw,'Xte_raw':Xte_raw,
        'ytr':ytr,'yte':yte,
        'pre':pre,'Xtr':Xtr,'Xte':Xte,
        'models':models
    }

pd.DataFrame(RESULTS).query("scenario == 'S0'").sort_values(['dataset','model'])

## 8. Scenario S1 — Imbalance Stress Test

S1 hanya diterapkan pada classifier supervised. Label hanya digunakan untuk membentuk training stress condition; test set tetap sama.

In [ ]:
# ============================================================
# 8. S1 IMBALANCE
# ============================================================
def rebalance_majority(X, y, minority_majority_ratio, seed=42):
    rng = np.random.RandomState(seed)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]

    # minority ditentukan berdasarkan jumlah aktual
    if len(idx0) <= len(idx1):
        min_idx, maj_idx = idx0, idx1
    else:
        min_idx, maj_idx = idx1, idx0

    target_maj = min(len(maj_idx), max(1, int(len(min_idx) / minority_majority_ratio)))
    maj_sel = rng.choice(maj_idx, target_maj, replace=False)

    keep = np.concatenate([min_idx, maj_sel])
    rng.shuffle(keep)
    return X[keep], y[keep]

for ds_name, c in CACHE.items():
    for ratio in CFG['S1_RATIOS']:
        Xtr_s, ytr_s = rebalance_majority(c['Xtr'], c['ytr'], ratio, CFG['SEED'])
        models = fit_supervised(Xtr_s, ytr_s)

        for mn, m in models.items():
            r = eval_supervised(m, c['Xte'], c['yte'])
            RESULTS.append({
                'dataset':ds_name,'scenario':'S1',
                'condition':f'minority_majority={ratio}',
                'model':mn, **r
            })

print("✅ S1 complete")

## 9. Scenario S2 — Noise dan Missing Telemetry

Perturbasi dilakukan pada **raw held-out features**, bukan pada training set.

- Noise Gaussian hanya diberikan pada fitur numerik.
- Besarnya noise relatif terhadap simpangan baku training.
- Missingness dimasukkan ke raw test lalu ditangani oleh imputer yang sudah di-fit pada training.

In [ ]:
# ============================================================
# 9. S2 NOISE + MISSING
# ============================================================
def perturb_numeric_raw(Xte_raw, Xtr_raw, sigma=0.0, missing_rate=0.0, seed=42):
    Xn = Xte_raw.copy()
    rng = np.random.RandomState(seed)

    num_cols = list(Xtr_raw.select_dtypes(include=np.number).columns)
    if not num_cols:
        return Xn

    for c in num_cols:
        tr = pd.to_numeric(Xtr_raw[c], errors='coerce')
        te = pd.to_numeric(Xn[c], errors='coerce')

        if sigma > 0:
            sd = float(np.nanstd(tr.values))
            if np.isfinite(sd) and sd > 1e-12:
                mask = te.notna().values
                vals = te.values.astype(float)
                vals[mask] += rng.normal(0, sigma*sd, mask.sum())
                Xn[c] = vals

        if missing_rate > 0:
            m = rng.rand(len(Xn)) < missing_rate
            Xn.loc[m, c] = np.nan

    return Xn

for ds_name, c in CACHE.items():
    # Noise
    for sigma in CFG['S2_NOISE_SIGMAS']:
        Xn_raw = perturb_numeric_raw(
            c['Xte_raw'], c['Xtr_raw'],
            sigma=sigma, missing_rate=0, seed=CFG['SEED']
        )
        Xn = c['pre'].transform(Xn_raw)
        for mn, m in c['models'].items():
            r = eval_supervised(m, Xn, c['yte'])
            RESULTS.append({
                'dataset':ds_name,'scenario':'S2_noise',
                'condition':f'sigma={sigma}','model':mn, **r
            })

    # Missing
    for miss in CFG['S2_MISSING_RATES']:
        Xm_raw = perturb_numeric_raw(
            c['Xte_raw'], c['Xtr_raw'],
            sigma=0, missing_rate=miss, seed=CFG['SEED']
        )
        Xm = c['pre'].transform(Xm_raw)
        for mn, m in c['models'].items():
            r = eval_supervised(m, Xm, c['yte'])
            RESULTS.append({
                'dataset':ds_name,'scenario':'S2_missing',
                'condition':f'missing={miss}','model':mn, **r
            })

print("✅ S2 complete")

## 10. Scenario S3 — Temporal/Order Shift

S3 menggunakan urutan temporal/urutan asli dataset.

Ini bukan cross-dataset generalization. Tujuannya adalah menguji apakah model yang belajar dari bagian awal data menurun pada bagian akhir.

Jika suatu dataset tidak memiliki timestamp eksplisit, notebook menggunakan urutan observasi/log sebagai proxy temporal dan hasil harus diberi label sesuai itu.

In [ ]:
# ============================================================
# 10. S3 TEMPORAL / ORDER SHIFT
# ============================================================
for ds_name, (X, y, time_key, meta) in DATASETS.items():
    order = np.argsort(np.asarray(time_key))
    n = len(order)
    cut = int(0.70 * n)
    tr_idx, te_idx = order[:cut], order[cut:]

    # perlu dua kelas di training dan test
    if len(np.unique(y[tr_idx])) < 2 or len(np.unique(y[te_idx])) < 2:
        print(f"⚠️ {ds_name}: S3 dilewati karena kelas tidak cukup pada temporal split.")
        continue

    Xtr_raw, Xte_raw = X.iloc[tr_idx].copy(), X.iloc[te_idx].copy()
    ytr, yte = y[tr_idx], y[te_idx]

    pre,_,_ = make_preprocessor(Xtr_raw)
    Xtr = pre.fit_transform(Xtr_raw)
    Xte = pre.transform(Xte_raw)

    models = fit_supervised(Xtr, ytr)
    for mn,m in models.items():
        r = eval_supervised(m, Xte, yte)
        RESULTS.append({
            'dataset':ds_name,'scenario':'S3_temporal',
            'condition':'first70_to_last30',
            'model':mn, **r
        })

print("✅ S3 complete")

## 11. Scenario S4 — Parameter Sensitivity pada Unsupervised Models

Tujuan S4 bukan mencari satu parameter terbaik universal, melainkan mengukur seberapa volatil alarm/FAR/performa terhadap perubahan parameter.

In [ ]:
# ============================================================
# 11. S4 PARAMETER SENSITIVITY
# ============================================================
for ds_name, c in CACHE.items():
    y_rate = float(np.clip(c['ytr'].mean(), 0.01, 0.20))

    for contam in CFG['IF_CONTAMS']:
        r = eval_iforest(c['Xtr'], c['Xte'], c['yte'], contam)
        RESULTS.append({
            'dataset':ds_name,'scenario':'S4_IF',
            'condition':f'contamination={contam}',
            'model':'IsolationForest', **r
        })

    for k in CFG['LOF_K']:
        r = eval_lof(c['Xtr'], c['Xte'], c['yte'], k, y_rate)
        RESULTS.append({
            'dataset':ds_name,'scenario':'S4_LOF',
            'condition':f'k={k}',
            'model':'LOF', **r
        })

    for q in CFG['DBSCAN_Q']:
        r, eps, ncore = eval_dbscan_inductive(c['Xtr'], c['Xte'], c['yte'], q, 10)
        RESULTS.append({
            'dataset':ds_name,'scenario':'S4_DBSCAN',
            'condition':f'eps_quantile={q}',
            'model':'DBSCAN_inductive',
            'eps':eps,'n_core':ncore, **r
        })

print("✅ S4 complete")

## 12. Scenario S5 — Core Inference Latency

S5 mengukur latency **predict() classifier**, bukan latency penuh termasuk SHAP.  
Karena itu hasil hanya boleh disebut *core inference feasibility*.

In [ ]:
# ============================================================
# 12. S5 LATENCY
# ============================================================
for ds_name, c in CACHE.items():
    for mn, m in c['models'].items():
        for bs in CFG['BATCH_SIZES']:
            n = min(bs, len(c['Xte']))
            Xb = c['Xte'][:n]

            # warm-up
            _ = m.predict(Xb[:min(4,n)])

            times = []
            for _ in range(20):
                t0 = time.perf_counter()
                _ = m.predict(Xb)
                times.append((time.perf_counter()-t0)*1000)

            RESULTS.append({
                'dataset':ds_name,'scenario':'S5_latency',
                'condition':f'batch={n}','model':mn,
                'latency_ms_mean':float(np.mean(times)),
                'latency_ms_p95':float(np.percentile(times,95)),
                'throughput_per_s':float(n/(np.mean(times)/1000))
            })

print("✅ S5 complete")

## 13. Confirmatory SHAP Stability Audit

Bagian ini hanya audit konfirmatori untuk melihat apakah alasan model berubah di bawah noise.

**Jangan disebut novelty Siklus 1.**  
FSS sebagai kontribusi metodologis tetap ditempatkan pada Siklus 2.

Output di sini hanya:
- Top-K Jaccard overlap,
- rank Spearman sederhana,
- beberapa noise seeds.

In [ ]:
# ============================================================
# 13. SHAP CONFIRMATORY AUDIT
# ============================================================
try:
    import shap
    from scipy.stats import spearmanr

    SHAP_ROWS = []

    def shap_matrix(model, X):
        ex = shap.TreeExplainer(model)
        sv = ex.shap_values(X)
        if isinstance(sv, list):
            sv = sv[1]
        sv = np.asarray(sv)
        if sv.ndim == 3:
            sv = sv[:,:,1]
        return sv.astype(float)

    def topk_jaccard(a, b, k=10):
        ma = np.abs(a).mean(0)
        mb = np.abs(b).mean(0)
        ia = set(np.argsort(-ma)[:k])
        ib = set(np.argsort(-mb)[:k])
        return len(ia & ib) / max(len(ia | ib), 1)

    for ds_name, c in CACHE.items():
        n = min(CFG['SHAP_N'], len(c['Xte']))
        idx = np.sort(np.random.RandomState(CFG['SEED']).choice(len(c['Xte']), n, replace=False))
        Xclean = c['Xte'][idx]

        for mn, model in c['models'].items():
            sv0 = shap_matrix(model, Xclean)
            base_imp = np.abs(sv0).mean(0)

            for seed in CFG['SHAP_SEEDS']:
                Xn_raw = perturb_numeric_raw(
                    c['Xte_raw'].iloc[idx].copy(),
                    c['Xtr_raw'],
                    sigma=CFG['SHAP_NOISE_SIGMA'],
                    missing_rate=0,
                    seed=seed
                )
                Xn = c['pre'].transform(Xn_raw)
                svn = shap_matrix(model, Xn)
                noisy_imp = np.abs(svn).mean(0)

                rho = spearmanr(base_imp, noisy_imp).statistic
                jac = topk_jaccard(sv0, svn, CFG['SHAP_K'])

                SHAP_ROWS.append({
                    'dataset':ds_name,'model':mn,'seed':seed,
                    'sigma':CFG['SHAP_NOISE_SIGMA'],
                    'topk_jaccard':jac,
                    'rank_spearman':rho
                })

    pd.DataFrame(SHAP_ROWS).to_csv(
        CFG['OUTPUT_DIR'] + 'shap_confirmatory_audit.csv', index=False
    )
    print("✅ SHAP audit complete")

except Exception as e:
    print("⚠️ SHAP audit skipped:", e)

## 14. Cross-Dataset Evidence Synthesis

Notebook tidak memberi label otomatis "knowledge-based W1–W6".

Sebagai gantinya, kita hitung evidence yang dapat dibandingkan:
- baseline F1,
- degradation pada noise/missing,
- temporal/order shift,
- volatilitas FAR/alert rate pada unsupervised sensitivity,
- latency.

Interpretasi W1–W6 dilakukan setelah melihat **recurrence across datasets**.

In [ ]:
# ============================================================
# 14. EVIDENCE SYNTHESIS
# ============================================================
res = pd.DataFrame(RESULTS)
res.to_csv(CFG['OUTPUT_DIR'] + 'siklus1_confirmatory_all_results.csv', index=False)

# baseline supervised
base = res[(res.scenario=='S0') & (res.model.isin(['XGBoost','RandomForest']))][
    ['dataset','model','f1','recall','far','pr_auc']
].rename(columns={
    'f1':'f1_s0','recall':'recall_s0','far':'far_s0','pr_auc':'pr_auc_s0'
})

# sigma=0.01 noise
noise = res[
    (res.scenario=='S2_noise') &
    (res.condition=='sigma=0.01') &
    (res.model.isin(['XGBoost','RandomForest']))
][['dataset','model','f1','recall','far']].rename(columns={
    'f1':'f1_noise_001','recall':'recall_noise_001','far':'far_noise_001'
})

syn = base.merge(noise, on=['dataset','model'], how='left')
syn['delta_f1_noise_001'] = syn['f1_noise_001'] - syn['f1_s0']
syn['pct_delta_f1_noise_001'] = 100 * syn['delta_f1_noise_001'] / syn['f1_s0'].replace(0,np.nan)

# temporal
temp = res[
    (res.scenario=='S3_temporal') &
    (res.model.isin(['XGBoost','RandomForest']))
][['dataset','model','f1','recall','far']].rename(columns={
    'f1':'f1_temporal','recall':'recall_temporal','far':'far_temporal'
})

syn = syn.merge(temp, on=['dataset','model'], how='left')

print("\n=== CROSS-DATASET CONFIRMATORY EVIDENCE ===")
display(syn.sort_values(['dataset','model']))

syn.to_csv(CFG['OUTPUT_DIR'] + 'cross_dataset_evidence_summary.csv', index=False)

# S4 volatility summary
s4 = res[res.scenario.isin(['S4_IF','S4_LOF','S4_DBSCAN'])].copy()
if len(s4):
    vol = s4.groupby(['dataset','model']).agg(
        far_min=('far','min'),
        far_max=('far','max'),
        alert_min=('alert_rate','min'),
        alert_max=('alert_rate','max'),
        f1_min=('f1','min'),
        f1_max=('f1','max')
    ).reset_index()
    vol['far_range'] = vol['far_max'] - vol['far_min']
    vol['alert_range'] = vol['alert_max'] - vol['alert_min']
    vol.to_csv(CFG['OUTPUT_DIR'] + 's4_parameter_volatility.csv', index=False)
    display(vol)

print("✅ Evidence files saved to:", CFG['OUTPUT_DIR'])

## 15. Interpretasi untuk Disertasi

Setelah notebook selesai dijalankan, sintesis penelitian **tidak boleh** langsung berbunyi:

> "Karena aturan X terjadi pada satu dataset, maka X adalah knowledge-based rule."

Yang dinilai adalah:

1. **Recurrence** — apakah weakness muncul pada lebih dari satu dataset?
2. **Consistency** — apakah arah degradasi konsisten walaupun besar efek berbeda?
3. **Boundary condition** — pada dataset apa weakness tidak muncul atau berlawanan?
4. **Model dependence** — apakah pola sama pada XGBoost dan Random Forest?
5. **Method dependence** — apakah unsupervised menunjukkan volatilitas berbeda?
6. **Operational implication** — weakness mana yang benar-benar membutuhkan treatment pada Siklus 2?

### Posisi metodologis yang aman

> Eksperimen Siklus 1 awal berpusat pada CICIDS2018. Setelah evaluasi metodologis dan masukan penguji, dilakukan confirmatory multi-dataset replication pada CICIDS2018, HDFS, BGL, dan UNSW-NB15. Confirmatory replication tidak mengubah tujuan RM1, tetapi menguji keberulangan dan batas generalisasi weakness W1–W6 sebelum hasilnya digunakan sebagai basis penurunan kebutuhan desain Siklus 2.

Dengan posisi tersebut, **experience** berasal dari banyak kondisi/dataset dan **knowledge** disusun sebagai pola empiris beserta boundary conditions, bukan sebagai rule universal dari satu kasus.